# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. The dataset examines protein abundance, modifications, and sequences from human samples.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print("Name:", metadata.name)
print("Description:", metadata.description)
print("Published on:", getattr(metadata, 'datePublished', 'N/A'))


## 2. Data Overview
Review available record sets and their `@id`s. We'll list the main data tables (`RecordSet`), and for each, enumerate their fields (columns) and their IDs.

In [ ]:
# List all RecordSets in the dataset using their `@id`
record_sets = list(ds.record_sets())
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name','N/A')}")
    print(f"  Description: {rs.get('description','N/A')}")
    print(f"  Fields (@id):")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            # Some Croissant schemas list actual field objects; others just list @id
            print(f"    - {field['@id']} : {field.get('name', '')}")
        else:
            print(f"    - {field}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Compile the list of RecordSet IDs (change if dataset updates)
record_set_ids = [rs['@id'] for rs in ds.record_sets()]
dataframes = {}

# Load each record set into a DataFrame
for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id} ...")
    records = list(ds.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(2))
    else:
        print('No records found.')
    print()


## 4. Exploratory Data Analysis (EDA)
We'll now demonstrate typical EDA steps on a main table in the dataset. We'll select a numeric field for further analysis. All field and record set references use their `@id` as per best practice.

In [ ]:
# Select a main record set (use ID printed above, for example):
main_record_set_id = None
for rid, df in dataframes.items():
    if len(df) > 0:
        main_record_set_id = rid
        break
assert main_record_set_id is not None, "No populated record set found!"

main_df = dataframes[main_record_set_id]
print(f"Exploring main record set: {main_record_set_id}")

# Show column names to assist in selecting a numeric field by @id
print("Fields (@id):", main_df.columns.tolist())

# Attempt to select a numeric field by common data column names or @id
numeric_field = None
for field in main_df.columns:
    if 'MW' in field or 'molecular_weight' in field or 'count' in field or 'coverage' in field or 'abundance' in field:
        numeric_field = field
        break
if numeric_field is None:
    numeric_field = main_df.select_dtypes(include=['number']).columns[0]

print(f"Using numeric field: {numeric_field}")

# Filter records with the numeric field above its median
if numeric_field in main_df.columns:
    threshold = main_df[numeric_field].median()
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f} (median): {len(filtered_df)} rows\n")

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a categorical field (e.g., 'description', 'accession', etc.)
    group_field = None
    for field in main_df.columns:
        if field.lower() in ['description', 'accession', 'category', 'group', 'sample']:
            group_field = field
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print(f"No suitable numeric field '{numeric_field}' found.")


## 5. Visualization
Let's visualize the distribution of the selected numeric field and relationship with the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field], kde=True)
plt.title(f"Distribution of {numeric_field} in record set {main_record_set_id}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If we found a group field, plot means by group (top groups only)
if 'group_field' in locals() and group_field and group_field in main_df.columns:
    means_by_group = main_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:10]
    plt.figure(figsize=(10, 4))
    sns.barplot(x=means_by_group.index.astype(str), y=means_by_group.values)
    plt.title(f"Mean {numeric_field} by {group_field} (top 10)")
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(f"{group_field}")
    plt.tight_layout()
    plt.show()


## 6. Conclusion
This notebook demonstrated the workflow for exploring a [FAIR^2 Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library:
- The dataset metadata was loaded directly from the Croissant schema.
- We listed record set and field `@id`s, extracted tabular data, and performed basic filtering and normalization.
- Exploratory data analysis and visualizations showed distributions and group-wise aggregated summaries.

**Next steps:** More domain-specific analyses can be performed, such as comparisons between sample groups, outlier detection, or linking to external knowledge bases using the accession IDs.